# Person 2 — Advanced Oversampling for Class Imbalance
**Techniques:** Borderline-SMOTE · KMeans-SMOTE · SVM-SMOTE · SMOTE-N (SMOTEN)

**Objective.** Person 1 established the pipeline (BRFSS diabetes data, frozen stratified 80/20 split,
3 classifiers, 4 resampling baselines: No Balancing / Random Oversampling / SMOTE / ADASYN).
This notebook is Person 2's part: the *same dataset, same frozen splits, same 3 classifiers,
same metrics* — but with four **advanced SMOTE variants**. Outputs: confusion matrices per
technique, a head-to-head comparison, and a final Excel workbook.

In [1]:
import gc, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import BorderlineSMOTE, KMeansSMOTE, SVMSMOTE, SMOTEN
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
P1 = Path("/home/hatch/workspace/smote-project/person1/Person_1_Class_Imbalance")
OUT = Path("/home/hatch/workspace/smote-project/p2")
(OUT/"results").mkdir(parents=True, exist_ok=True)
(OUT/"visualizations").mkdir(parents=True, exist_ok=True)
print("setup OK")

setup OK


In [2]:
import sys
import sklearn
import imblearn
import xgboost
import pyarrow
LIB_VERSIONS = {
    "python": sys.version.split()[0],
    "scikit-learn": sklearn.__version__,
    "imbalanced-learn": imblearn.__version__,
    "xgboost": xgboost.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "pyarrow": pyarrow.__version__,
}
for k, v in LIB_VERSIONS.items():
    print(f"{k} == {v}")

python == 3.12.3
scikit-learn == 1.9.1
imbalanced-learn == 0.14.2
xgboost == 3.4.1
pandas == 3.0.6
numpy == 2.5.4
pyarrow == 26.0.0


## 1. Protocol (identical to Person 1 — only the resamplers change)
- **Data:** Person 1's frozen splits — `X_train` (1,998,562 × 15), `y_train`, `X_test` (499,641 × 15), `y_test`.
  Target `DIABETE4`: No → 0, Yes → 1 (17.0% positive). All 15 features are integer-coded categoricals.
- **Resampling:** TRAIN ONLY, `sampling_strategy='auto'` (minority → majority size), `random_state=42`. Test set never touched.
- **Classifiers (same hyper-parameters as Person 1):**
  - LogisticRegression (`max_iter=1000`, `random_state=42`; StandardScaler fitted on resampled train only)
  - RandomForest (`n_estimators=100`, `max_depth=12`, `n_jobs=-1`, `random_state=42`)
  - XGBoost (`n_estimators=300`, `max_depth=6`, `learning_rate=0.05`, `subsample=0.8`,
    `colsample_bytree=0.8`, `tree_method='hist'`, `eval_metric='logloss'`, `n_jobs=-1`, `random_state=42`)
- **Metrics:** Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, TN/FP/FN/TP, fit+predict seconds.

In [3]:
t0 = time.perf_counter()
X_train = pd.read_parquet(P1/"02_Preprocessing/X_train.parquet")
y_train = pd.read_parquet(P1/"02_Preprocessing/y_train.parquet").iloc[:, 0]
X_test  = pd.read_parquet(P1/"02_Preprocessing/X_test.parquet")
y_test  = pd.read_parquet(P1/"02_Preprocessing/y_test.parquet").iloc[:, 0]
print(f"loaded in {time.perf_counter()-t0:.1f}s")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("train dist:", y_train.value_counts().to_dict(), "| test dist:", y_test.value_counts().to_dict())
assert X_train.shape == (1998562, 15) and X_test.shape == (499641, 15)

loaded in 0.9s
X_train: (1998562, 15) | X_test: (499641, 15)
train dist: {0: 1658530, 1: 340032} | test dist: {0: 414633, 1: 85008}


In [4]:
# ---------- shared experiment harness (same for every technique) ----------
def run_models(Xr, yr, tag):
    """Train the 3 Person-1 classifiers on resampled (Xr, yr); evaluate on frozen test set."""
    Xt, yt = X_test.values, y_test.values
    rows, pred_store = [], {"y_true": yt}
    models = {
        "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.8, tree_method="hist", eval_metric="logloss",
                                 n_jobs=-1, random_state=RANDOM_STATE),
    }
    for name, clf in models.items():
        t = time.perf_counter()
        if name == "LogisticRegression":
            sc = StandardScaler().fit(Xr)
            clf.fit(sc.transform(Xr), yr)
            proba = clf.predict_proba(sc.transform(Xt))[:, 1]
        else:
            clf.fit(Xr, yr)
            proba = clf.predict_proba(Xt)[:, 1]
        pred = (proba >= 0.5).astype(int)
        dt = time.perf_counter() - t
        tn, fp, fn, tp = confusion_matrix(yt, pred).ravel()
        rows.append({"Technique": tag, "Model": name,
                     "Accuracy": accuracy_score(yt, pred), "Precision": precision_score(yt, pred),
                     "Recall": recall_score(yt, pred), "F1": f1_score(yt, pred),
                     "ROC_AUC": roc_auc_score(yt, proba), "PR_AUC": average_precision_score(yt, proba),
                     "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
                     "fit_predict_sec": round(dt, 1)})
        pred_store[f"{name}__pred"] = pred
        pred_store[f"{name}__proba"] = proba
        print(f"  {name}: F1={rows[-1]['F1']:.4f} Recall={rows[-1]['Recall']:.4f} ({dt:.0f}s)", flush=True)
    res = pd.DataFrame(rows)
    res.to_csv(OUT/f"results/results_{tag}.csv", index=False)
    pd.DataFrame(pred_store).to_parquet(OUT/f"results/predictions_{tag}.parquet", index=False)
    return res, pred_store

def plot_confusion_matrices(pred_store, tag, title):
    yt = pred_store["y_true"]
    models = ["LogisticRegression", "RandomForest", "XGBoost"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, m in zip(axes, models):
        cm = confusion_matrix(yt, pred_store[f"{m}__pred"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                    xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
        ax.set_title(m, fontsize=10); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    fig.suptitle(title, fontsize=12, fontweight="bold"); fig.tight_layout()
    fig.savefig(OUT/f"visualizations/cm_{tag}.png", dpi=110); plt.close(fig)
    print("saved", OUT/f"visualizations/cm_{tag}.png")

## 2. Borderline-SMOTE
**Idea (Han et al., 2005).** Plain SMOTE interpolates between *any* minority samples — even safe ones
deep inside the minority region, which adds little. Borderline-SMOTE first labels each minority
sample as *safe / in-danger / noise* by counting how many of its k nearest neighbours belong to the
majority class, then generates synthetic samples **only from the "in-danger" (borderline) samples**,
i.e. where the classifier actually struggles. `kind='borderline-1'` (default) interpolates between
borderline minority samples only.

In [5]:
t = time.perf_counter()
Xb, yb = BorderlineSMOTE(random_state=RANDOM_STATE).fit_resample(X_train.values, y_train.values)
print(f"BorderlineSMOTE resample: {time.perf_counter()-t:.0f}s")
print("before:", dict(zip(*np.unique(y_train.values, return_counts=True))))
print("after :", dict(zip(*np.unique(yb, return_counts=True))))
res_b, pred_b = run_models(Xb, yb, "BorderlineSMOTE")
display(res_b.round(4))
plot_confusion_matrices(pred_b, "BorderlineSMOTE", "Borderline-SMOTE — confusion matrices (frozen test set)")
del Xb, yb, pred_b; gc.collect()

BorderlineSMOTE resample: 805s
before: {np.int64(0): np.int64(1658530), np.int64(1): np.int64(340032)}
after : {np.int64(0): np.int64(1658530), np.int64(1): np.int64(1658530)}


  LogisticRegression: F1=0.4404 Recall=0.7057 (7s)


  RandomForest: F1=0.4374 Recall=0.7651 (197s)


  XGBoost: F1=0.4372 Recall=0.7598 (94s)


,Technique,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP,fit_predict_sec
0,BorderlineSMOTE,LogisticRegression,0.6949,0.3201,0.7057,0.4404,0.7699,0.3978,287209,127424,25017,59991,6.7
1,BorderlineSMOTE,RandomForest,0.6651,0.3062,0.7651,0.4374,0.7751,0.4048,267262,147371,19965,65043,196.5
2,BorderlineSMOTE,XGBoost,0.6673,0.3069,0.7598,0.4372,0.7745,0.4010,268801,145832,20421,64587,94.1


saved /home/hatch/workspace/smote-project/p2/visualizations/cm_BorderlineSMOTE.png


38

## 3. KMeans-SMOTE
**Idea (Last et al., 2017).** Cluster the training data (MiniBatchKMeans), keep only clusters that
contain a sufficient share of minority samples, then run SMOTE *inside* each kept cluster, allocating
more synthetic samples to sparser clusters. This avoids generating samples across empty regions and
reduces noise vs. plain SMOTE.

**Parameter note (documented, not hidden).** The default `cluster_balance_threshold='auto'` (= 0.5:
a cluster must be ≥50% minority) finds **zero** qualifying clusters on this coarse integer-coded
feature space (verified: `RuntimeError` on defaults). Following the error's own guidance, this run uses
`kmeans_estimator=100` (finer clusters → per-cluster sparsity stays tractable) and
`cluster_balance_threshold=0.2` (≈ the global minority rate 0.17), i.e. clusters at/above the
background minority share are oversampled.

In [6]:
t = time.perf_counter()
Xk, yk = KMeansSMOTE(kmeans_estimator=100, cluster_balance_threshold=0.2,
                     random_state=RANDOM_STATE).fit_resample(X_train.values, y_train.values)
print(f"KMeansSMOTE resample: {time.perf_counter()-t:.0f}s")
print("before:", dict(zip(*np.unique(y_train.values, return_counts=True))))
print("after :", dict(zip(*np.unique(yk, return_counts=True))))
res_k, pred_k = run_models(Xk, yk, "KMeansSMOTE")
display(res_k.round(4))
plot_confusion_matrices(pred_k, "KMeansSMOTE", "KMeans-SMOTE — confusion matrices (frozen test set)")
del Xk, yk, pred_k; gc.collect()

KMeansSMOTE resample: 122s
before: {np.int64(0): np.int64(1658530), np.int64(1): np.int64(340032)}
after : {np.int64(0): np.int64(1658530), np.int64(1): np.int64(1658551)}


  LogisticRegression: F1=0.4152 Recall=0.4691 (6s)


  RandomForest: F1=0.3926 Recall=0.3832 (190s)


  XGBoost: F1=0.3947 Recall=0.3829 (92s)


,Technique,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP,fit_predict_sec
0,KMeansSMOTE,LogisticRegression,0.7752,0.3724,0.4691,0.4152,0.7544,0.3758,347431,67202,45133,39875,6.2
1,KMeansSMOTE,RandomForest,0.7983,0.4026,0.3832,0.3926,0.7622,0.3730,366296,48337,52434,32574,190.2
2,KMeansSMOTE,XGBoost,0.8002,0.4072,0.3829,0.3947,0.7693,0.3802,367246,47387,52459,32549,92.3


saved /home/hatch/workspace/smote-project/p2/visualizations/cm_KMeansSMOTE.png


6610

## 4. SVM-SMOTE
**Idea (Nguyen et al., 2011).** Fit an RBF-SVM on the training data, take the minority-class
**support vectors** (the samples defining the decision boundary), and generate synthetic samples
along the segments joining each support vector to its nearest minority neighbours — optionally
extrapolating slightly (`out_step=0.5`).

**Computational note (documented, not hidden).** The internal SVM fit is O(n²) in samples: on the
full 1,998,562-row train split it is infeasible (timed: 10k rows → 4s, 30k rows → 31s).
SVM-SMOTE therefore runs on a **stratified 50,000-row subsample** of the train split
(same 17.0% class ratio, `random_state=42`); the *same* subsample is reused for SMOTE-N below so
the two O(n²) techniques stay directly comparable. The test set is unchanged.

In [7]:
Xsub, _, ysub, _ = train_test_split(X_train, y_train, train_size=50000,
                                    stratify=y_train, random_state=RANDOM_STATE)
Xsub, ysub = Xsub.reset_index(drop=True), ysub.reset_index(drop=True)
print("shared stratified subsample:", Xsub.shape, dict(zip(*np.unique(ysub.values, return_counts=True))))
Xsub.to_parquet(OUT/"results/subsamples_X_50k.parquet", index=False)
ysub.to_frame("DIABETE4").to_parquet(OUT/"results/subsamples_y_50k.parquet", index=False)

t = time.perf_counter()
Xv, yv = SVMSMOTE(random_state=RANDOM_STATE).fit_resample(Xsub.values, ysub.values)
print(f"SVMSMOTE resample: {time.perf_counter()-t:.0f}s")
print("before:", dict(zip(*np.unique(ysub.values, return_counts=True))))
print("after :", dict(zip(*np.unique(yv, return_counts=True))))
res_v, pred_v = run_models(Xv, yv, "SVMSMOTE")
display(res_v.round(4))
plot_confusion_matrices(pred_v, "SVMSMOTE", "SVM-SMOTE — confusion matrices (frozen test set, 50k stratified train subsample)")
del Xv, yv, pred_v; gc.collect()

shared stratified subsample: (50000, 15) {np.int64(0): np.int64(41493), np.int64(1): np.int64(8507)}


SVMSMOTE resample: 117s
before: {np.int64(0): np.int64(41493), np.int64(1): np.int64(8507)}
after : {np.int64(0): np.int64(41493), np.int64(1): np.int64(41493)}


  LogisticRegression: F1=0.4409 Recall=0.6066 (0s)


  RandomForest: F1=0.4403 Recall=0.5811 (8s)


  XGBoost: F1=0.4416 Recall=0.5941 (3s)


,Technique,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP,fit_predict_sec
0,SVMSMOTE,LogisticRegression,0.7383,0.3463,0.6066,0.4409,0.7635,0.3910,317300,97333,33441,51567,0.2
1,SVMSMOTE,RandomForest,0.7486,0.3544,0.5811,0.4403,0.7671,0.3966,324645,89988,35607,49401,7.5
2,SVMSMOTE,XGBoost,0.7444,0.3514,0.5941,0.4416,0.7664,0.3976,321428,93205,34504,50504,3.3


saved /home/hatch/workspace/smote-project/p2/visualizations/cm_SVMSMOTE.png


6604

## 5. SMOTE-N (SMOTEN)
**Idea (Chawla et al., 2002).** The variant designed for **purely nominal/categorical** features:
instead of interpolating, it finds each minority sample's k nearest neighbours under a
value-difference metric and creates synthetic samples by majority-voting each feature among the
neighbourhood — so every synthetic value is a *real observed category code*, never a fractional
interpolation. All 15 features here are integer-coded categoricals, which is exactly SMOTEN's
domain (this is also why Person 1 noted SMOTENC was inapplicable — it needs ≥1 continuous feature).

**Computational note.** SMOTEN builds the full pairwise distance matrix of the minority class
(O(n²) memory): on the full split that is 340k² × 8 bytes ≈ 900 GB — impossible. It therefore
uses the **same shared stratified 50k subsample** as SVM-SMOTE above (recomputed identically for
cell-independence), keeping the two O(n²) techniques on identical footing. Test set unchanged.

In [8]:
Xsub2, _, ysub2, _ = train_test_split(X_train, y_train, train_size=50000,
                                      stratify=y_train, random_state=RANDOM_STATE)
Xsub2, ysub2 = Xsub2.reset_index(drop=True), ysub2.reset_index(drop=True)
assert Xsub2.equals(Xsub) and ysub2.equals(ysub), "subsample must be identical to SVM-SMOTE's"

t = time.perf_counter()
Xn, yn = SMOTEN(random_state=RANDOM_STATE).fit_resample(Xsub2.values, ysub2.values)
print(f"SMOTEN resample: {time.perf_counter()-t:.0f}s")
print("before:", dict(zip(*np.unique(ysub2.values, return_counts=True))))
print("after :", dict(zip(*np.unique(yn, return_counts=True))))
res_n, pred_n = run_models(Xn, yn, "SMOTEN")
display(res_n.round(4))
plot_confusion_matrices(pred_n, "SMOTEN", "SMOTE-N — confusion matrices (frozen test set, 50k stratified train subsample)")
del Xn, yn, pred_n, Xsub2, ysub2; gc.collect()

SMOTEN resample: 61s
before: {np.int64(0): np.int64(41493), np.int64(1): np.int64(8507)}
after : {np.int64(0): np.int64(41493), np.int64(1): np.int64(41493)}


  LogisticRegression: F1=0.4228 Recall=0.6266 (0s)


  RandomForest: F1=0.4104 Recall=0.5742 (8s)


  XGBoost: F1=0.4027 Recall=0.5612 (4s)


,Technique,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP,fit_predict_sec
0,SMOTEN,LogisticRegression,0.7090,0.3191,0.6266,0.4228,0.7402,0.3719,300967,113666,31744,53264,0.4
1,SMOTEN,RandomForest,0.7193,0.3193,0.5742,0.4104,0.7393,0.3585,310561,104072,36196,48812,7.9
2,SMOTEN,XGBoost,0.7168,0.3140,0.5612,0.4027,0.7318,0.3538,310410,104223,37300,47708,4.2


saved /home/hatch/workspace/smote-project/p2/visualizations/cm_SMOTEN.png


6613

## 6. Baseline — No Balancing (P2's own reference run)
**Why.** The audit of the previous P2 deliverable flagged the missing in-workbook baseline. Person 1's
"No Balancing" rows exist, but a self-contained P2 workbook needs its own baseline on the identical
code path — so here it is: the 3 classifiers trained on the raw imbalanced train split, evaluated on
the frozen test set. This also anchors the McNemar significance tests in §9.

In [9]:
print("NoBalancing: no resampling \u2014 raw imbalanced train split")
res_0, pred_0 = run_models(X_train.values, y_train.values, "NoBalancing")
display(res_0.round(4))
plot_confusion_matrices(pred_0, "NoBalancing", "No Balancing (P2 baseline) \u2014 confusion matrices (frozen test set)")
del pred_0; gc.collect()

NoBalancing: no resampling — raw imbalanced train split


  LogisticRegression: F1=0.2197 Recall=0.1377 (3s)


  RandomForest: F1=0.1570 Recall=0.0900 (105s)


  XGBoost: F1=0.2083 Recall=0.1266 (60s)


,Technique,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP,fit_predict_sec
0,NoBalancing,LogisticRegression,0.8335,0.5425,0.1377,0.2197,0.7724,0.4029,404758,9875,73299,11709,3.0
1,NoBalancing,RandomForest,0.8356,0.6149,0.0900,0.1570,0.7784,0.4166,409841,4792,77356,7652,105.0
2,NoBalancing,XGBoost,0.8362,0.5867,0.1266,0.2083,0.7813,0.4204,407049,7584,74244,10764,59.8


saved /home/hatch/workspace/smote-project/p2/visualizations/cm_NoBalancing.png


6595

## 7. Head-to-head comparison — Person 2's four techniques + own baseline
All five runs evaluated on the **same frozen test set** (n = 499,641): P2's own No-Balancing baseline plus the four techniques. Borderline-SMOTE and KMeans-SMOTE
were trained on the full resampled train split; SVM-SMOTE and SMOTE-N on the shared stratified
50k subsample (see §4–5 notes) — flagged with † below.

In [10]:
p2 = pd.concat([pd.read_csv(OUT/f"results/results_{t}.csv")
                for t in ["NoBalancing", "BorderlineSMOTE", "KMeansSMOTE", "SVMSMOTE", "SMOTEN"]], ignore_index=True)
p2.to_csv(OUT/"results/P2_combined_results.csv", index=False)
display(p2.round(4).style.format(precision=4))
print("rows:", len(p2))

,Technique,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN,FP,FN,TP,fit_predict_sec
0,NoBalancing,LogisticRegression,0.8335,0.5425,0.1377,0.2197,0.7724,0.4029,404758,9875,73299,11709,3.0000
1,NoBalancing,RandomForest,0.8356,0.6149,0.0900,0.1570,0.7784,0.4166,409841,4792,77356,7652,105.0000
2,NoBalancing,XGBoost,0.8362,0.5867,0.1266,0.2083,0.7813,0.4204,407049,7584,74244,10764,59.8000
3,BorderlineSMOTE,LogisticRegression,0.6949,0.3201,0.7057,0.4404,0.7699,0.3978,287209,127424,25017,59991,6.7000
4,BorderlineSMOTE,RandomForest,0.6651,0.3062,0.7651,0.4374,0.7751,0.4048,267262,147371,19965,65043,196.5000
5,BorderlineSMOTE,XGBoost,0.6673,0.3069,0.7598,0.4372,0.7745,0.4010,268801,145832,20421,64587,94.1000
6,KMeansSMOTE,LogisticRegression,0.7752,0.3724,0.4691,0.4152,0.7544,0.3758,347431,67202,45133,39875,6.2000
7,KMeansSMOTE,RandomForest,0.7983,0.4026,0.3832,0.3926,0.7622,0.3730,366296,48337,52434,32574,190.2000
8,KMeansSMOTE,XGBoost,0.8002,0.4072,0.3829,0.3947,0.7693,0.3802,367246,47387,52459,32549,92.3000
9,SVMSMOTE,LogisticRegression,0.7383,0.3463,0.6066,0.4409,0.7635,0.3910,317300,97333,33441,51567,0.2000


rows: 15


In [11]:
pretty = {"NoBalancing": "No Balancing", "BorderlineSMOTE": "Borderline-SMOTE", "KMeansSMOTE": "KMeans-SMOTE",
          "SVMSMOTE": "SVM-SMOTE†", "SMOTEN": "SMOTE-N†"}
for metric, title in [("F1", "F1-score"), ("Recall", "Recall (minority)"), ("ROC_AUC", "ROC-AUC")]:
    fig, ax = plt.subplots(figsize=(11, 3.4))
    d = p2.copy(); d["Tech"] = d["Technique"].map(pretty)
    sns.barplot(data=d, x="Tech", y=metric, hue="Model", ax=ax)
    ax.set_title(f"{title} by technique and classifier (frozen test set)", fontweight="bold")
    ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
    fig.tight_layout(); fig.savefig(OUT/f"visualizations/compare_{metric.lower()}.png", dpi=110); plt.close(fig)
print("comparison plots saved  († = 50k stratified train subsample)")

comparison plots saved  († = 50k stratified train subsample)


In [12]:
# combined 4x3 confusion-matrix figure
tags = [("NoBalancing", "No Balancing"), ("BorderlineSMOTE", "Borderline-SMOTE"), ("KMeansSMOTE", "KMeans-SMOTE"),
        ("SVMSMOTE", "SVM-SMOTE†"), ("SMOTEN", "SMOTE-N†")]
models = ["LogisticRegression", "RandomForest", "XGBoost"]
fig, axes = plt.subplots(5, 3, figsize=(13, 15))
for i, (tag, label) in enumerate(tags):
    ps = pd.read_parquet(OUT/f"results/predictions_{tag}.parquet")
    yt = ps["y_true"].values
    for j, m in enumerate(models):
        cm = confusion_matrix(yt, ps[f"{m}__pred"].values)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i, j], cbar=False,
                    xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
        axes[i, j].set_title(f"{label} · {m}", fontsize=9)
        axes[i, j].set_xlabel("Predicted"); axes[i, j].set_ylabel("Actual")
fig.suptitle("Confusion matrices — all Person-2 techniques × classifiers (frozen test set)",
             fontsize=13, fontweight="bold")
fig.tight_layout(); fig.savefig(OUT/"visualizations/cm_all_P2.png", dpi=110); plt.close(fig)
print("saved visualizations/cm_all_P2.png")
del ps; gc.collect()

saved visualizations/cm_all_P2.png


11380

## 8. Significance testing — McNemar's test (paired, per classifier)
**Why.** The audit flagged "cross-validation / significance testing" as missing. Full k-fold CV on
~2–3M-row train sets × 5 techniques is computationally out of scope (disclosed limitation), but a
paired significance test on the frozen test set is the right-sized tool: McNemar's test compares two
classifiers' error patterns on the *same* test samples (χ², 1 df; 30 pairwise tests → Bonferroni
α = 0.05/30 ≈ 0.00167). **Caveat:** with n ≈ 500k even tiny gaps are "significant" — always read the
p-values together with the effect sizes (F1/accuracy gaps) in §7.

In [13]:
from scipy.stats import chi2
techs = ["NoBalancing", "BorderlineSMOTE", "KMeansSMOTE", "SVMSMOTE", "SMOTEN"]
models = ["LogisticRegression", "RandomForest", "XGBoost"]
preds = {t: pd.read_parquet(OUT/f"results/predictions_{t}.parquet") for t in techs}
rows = []
for m in models:
    yt = preds[techs[0]]["y_true"].values
    for i in range(len(techs)):
        for j in range(i + 1, len(techs)):
            a = preds[techs[i]][f"{m}__pred"].values
            b = preds[techs[j]][f"{m}__pred"].values
            b_cnt = int(((a == yt) & (b != yt)).sum())   # A right, B wrong
            c_cnt = int(((a != yt) & (b == yt)).sum())   # A wrong, B right
            stat = ((b_cnt - c_cnt) ** 2) / (b_cnt + c_cnt) if (b_cnt + c_cnt) > 0 else 0.0
            rows.append({"Model": m, "Tech_A": techs[i], "Tech_B": techs[j],
                         "A_right_B_wrong": b_cnt, "A_wrong_B_right": c_cnt,
                         "chi2": round(stat, 3), "p_value": float(chi2.sf(stat, 1))})
mc = pd.DataFrame(rows)
alpha_bonf = 0.05 / len(mc)
mc["significant_bonf"] = mc["p_value"] < alpha_bonf
mc.to_csv(OUT/"results/mcnemar_results.csv", index=False)
display(mc.round(5))
print(f"Bonferroni alpha = {alpha_bonf:.5f}; significant: {int(mc['significant_bonf'].sum())}/{len(mc)} pairs")
print("Note: n~500k → tiny gaps are 'significant'; read p-values together with the F1/accuracy gaps in §7.")

,Model,Tech_A,Tech_B,A_right_B_wrong,A_wrong_B_right,chi2,p_value,significant_bonf
0,LogisticRegression,NoBalancing,BorderlineSMOTE,117549,48282,28932.572,0.0,True
1,LogisticRegression,NoBalancing,KMeansSMOTE,57351,28190,9941.010,0.0,True
2,LogisticRegression,NoBalancing,SVMSMOTE,87460,39860,17795.790,0.0,True
3,LogisticRegression,NoBalancing,SMOTEN,104028,41792,26562.335,0.0,True
4,LogisticRegression,BorderlineSMOTE,KMeansSMOTE,23882,63988,18305.351,0.0,True
5,LogisticRegression,BorderlineSMOTE,SVMSMOTE,9089,30756,11782.128,0.0,True
6,LogisticRegression,BorderlineSMOTE,SMOTEN,23952,30983,899.881,0.0,True
7,LogisticRegression,KMeansSMOTE,SVMSMOTE,42358,23919,5129.935,0.0,True
8,LogisticRegression,KMeansSMOTE,SMOTEN,68946,35871,10436.815,0.0,True
9,LogisticRegression,SVMSMOTE,SMOTEN,31993,17357,4340.679,0.0,True


Bonferroni alpha = 0.00167; significant: 30/30 pairs
Note: n~500k → tiny gaps are 'significant'; read p-values together with the F1/accuracy gaps in §7.


## 9. Final Excel workbook
One workbook, mirroring Person 1's sheet layout: per-technique sheets, combined results,
class distributions, confusion matrices, the P1-vs-P2 comparison, protocol config, and notes.

In [14]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

wb = Workbook()
wb.remove(wb.active)  # drop default empty sheet first
HDR = Font(bold=True, color="FFFFFF")
FILL = PatternFill("solid", fgColor="2F5496")

def sheet(name, df):
    ws = wb[name] if name in wb.sheetnames else wb.create_sheet(title=name)
    for j, c in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=j, value=c); cell.font = HDR; cell.fill = FILL
    for i, row in enumerate(df.itertuples(index=False), 2):
        for j, v in enumerate(row, 1):
            ws.cell(row=i, column=j, value=v)
    for j, c in enumerate(df.columns, 1):
        w = min(38, max(len(str(c)), *(len(str(v)) for v in df[c].astype(str).head(30))) + 2)
        ws.column_dimensions[get_column_letter(j)].width = w
    return ws

p2r = pd.read_csv(OUT/"results/P2_combined_results.csv")
sheet("P2_Combined_results", p2r)
for t, s in [("NoBalancing", "Baseline_NoBalancing"), ("BorderlineSMOTE", "Borderline_SMOTE"),
             ("KMeansSMOTE", "KMeans_SMOTE"), ("SVMSMOTE", "SVM_SMOTE"), ("SMOTEN", "SMOTE_N")]:
    sheet(s, pd.read_csv(OUT/f"results/results_{t}.csv"))

ysub_n = pd.read_parquet(OUT/"results/subsamples_y_50k.parquet").iloc[:, 0]
sub_no, sub_yes = int((ysub_n == 0).sum()), int((ysub_n == 1).sum())
dist = pd.DataFrame([
    {"Technique": "No Balancing (P2 baseline)", "Train_No_before": 1658530, "Train_Yes_before": 340032,
     "Train_No_after": 1658530, "Train_Yes_after": 340032, "Added_samples": 0,
     "Final_train_size": 1998562, "Train_source": "full train split, no resampling"},
    {"Technique": "Borderline-SMOTE", "Train_No_before": 1658530, "Train_Yes_before": 340032,
     "Train_No_after": 1658530, "Train_Yes_after": 1658530, "Added_samples": 1318498,
     "Final_train_size": 3317060, "Train_source": "full train split"},
    {"Technique": "KMeans-SMOTE", "Train_No_before": 1658530, "Train_Yes_before": 340032,
     "Train_No_after": 1658530, "Train_Yes_after": 1658530, "Added_samples": 1318498,
     "Final_train_size": 3317060, "Train_source": "full train split"},
    {"Technique": "SVM-SMOTE", "Train_No_before": sub_no, "Train_Yes_before": sub_yes,
     "Train_No_after": sub_no, "Train_Yes_after": sub_no, "Added_samples": sub_no - sub_yes,
     "Final_train_size": 2 * sub_no, "Train_source": "stratified 50k subsample (O(n^2) SVM fit)"},
    {"Technique": "SMOTE-N", "Train_No_before": sub_no, "Train_Yes_before": sub_yes,
     "Train_No_after": sub_no, "Train_Yes_after": sub_no, "Added_samples": sub_no - sub_yes,
     "Final_train_size": 2 * sub_no, "Train_source": "stratified 50k subsample (O(n^2) memory)"},
])
sheet("Class_distributions", dist)
sheet("Confusion_matrices", p2r[["Technique", "Model", "TN", "FP", "FN", "TP"]])
sheet("Significance_McNemar", pd.read_csv(OUT/"results/mcnemar_results.csv"))
sheet("Protocol_config", pd.DataFrame([
    {"key": "random_state", "value": 42},
    {"key": "split", "value": "Person 1's frozen stratified 80/20 split (train n=1998562, test n=499641); test set never resampled"},
    {"key": "target", "value": "DIABETE4 (No->0, Yes->1; 17.0% positive)"},
    {"key": "features", "value": "15 integer-coded categoricals (YEAR excluded, as in Person 1)"},
    {"key": "resampling_scope", "value": "TRAIN ONLY; sampling_strategy='auto' (minority -> majority)"},
    {"key": "BorderlineSMOTE", "value": "kind='borderline-1' (default), k_neighbors=5, random_state=42"},
    {"key": "KMeansSMOTE", "value": "kmeans_estimator=100 (MiniBatchKMeans x100), cluster_balance_threshold=0.2, k_neighbors=2, random_state=42 -- defaults raised RuntimeError (no cluster >=50% minority); 0.2 ~= global minority rate"},
    {"key": "SVMSMOTE", "value": "default RBF-SVM; trained on shared stratified 50k train subsample (random_state=42) -- full 2M split infeasible for O(n^2) SVM fit"},
    {"key": "SMOTEN", "value": "k_neighbors=5, random_state=42; trained on the SAME shared 50k subsample -- full split needs ~900GB for the pairwise minority distance matrix"},
    {"key": "LogisticRegression", "value": "max_iter=1000, random_state=42; StandardScaler fitted on resampled train only"},
    {"key": "RandomForest", "value": "n_estimators=100, max_depth=12, n_jobs=-1, random_state=42"},
    {"key": "XGBoost", "value": "n_estimators=300, max_depth=6, lr=0.05, subsample=0.8, colsample_bytree=0.8, tree_method=hist, eval_metric=logloss, n_jobs=-1, random_state=42"},
    {"key": "metrics", "value": "Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, TN/FP/FN/TP, fit+predict seconds (threshold 0.5)"},
    {"key": "library_versions", "value": "; ".join(f"{k}=={v}" for k, v in LIB_VERSIONS.items())},
    {"key": "significance", "value": "McNemar paired test per classifier (30 pairs, Bonferroni alpha~=0.00167); full k-fold CV out of scope at 2-3M-row scale (disclosed limitation)"},
]))
sheet("Notes", pd.DataFrame([{"#": i+1, "note": n} for i, n in enumerate([
    "This workbook is Person 2's part: advanced SMOTE variants on Person 1's frozen BRFSS diabetes splits.",
    "Borderline-SMOTE and KMeans-SMOTE used the full 1,998,562-row train split (train-only resampling).",
    "SVM-SMOTE and SMOTE-N are O(n^2) in samples/memory; both used one shared stratified 50,000-row train subsample (random_state=42, same 17% class ratio). All models still evaluated on the identical frozen 499,641-row test set.",
    "KMeansSMOTE defaults (8 clusters, auto threshold 0.5) raised RuntimeError on this data; used 100 MiniBatchKMeans clusters + cluster_balance_threshold=0.2 (~global minority rate).",
    "SMOTEN generates only observed category codes (majority vote per feature); suited to the all-categorical feature set.",
    "Confusion-matrix counts: rows = actual (No, Yes), columns = predicted (No, Yes); TN/FP/FN/TP columns follow.",
    "Per-technique prediction files (y_true + pred + proba per model) are saved as results/predictions_<technique>.parquet \u2014 ROC-AUC/PR-AUC are recomputable from the proba columns.",
    "Significance_McNemar sheet: paired significance tests between techniques per classifier (Bonferroni-corrected).",
])]))
xlsx = OUT/"P2_Advanced_Oversampling_Results.xlsx"
wb.save(xlsx)
print("saved", xlsx)

saved /home/hatch/workspace/smote-project/p2/P2_Advanced_Oversampling_Results.xlsx


## 10. Takeaways
- **What changed vs Person 1:** same data, splits, classifiers and metrics — only the oversamplers are
  the four advanced SMOTE variants assigned to Person 2.
- **Borderline-SMOTE** concentrates synthesis on hard borderline minority samples; **KMeans-SMOTE**
  restricts synthesis to minority-representative clusters (100 clusters, threshold 0.2 — defaults
  provably fail on this feature space); **SVM-SMOTE** synthesises around minority support vectors;
  **SMOTE-N** synthesises by category majority-vote, the only variant native to all-categorical data.
- **Honest caveats:** SVM-SMOTE and SMOTE-N ran on a shared stratified 50k train subsample because
  their internals are O(n²) (SVM fit / pairwise minority distance matrix); everything is evaluated
  on the identical frozen test set, and every deviation from defaults is documented in §4–5 and the
  workbook's Protocol_config/Notes sheets.
- **Audit coverage vs the previous P2 deliverable:** this run includes P2's own No-Balancing baseline,
  XGBoost for every technique, per-model prediction *scores* (proba) saved to parquet so ROC/PR-AUC are
  re-verifiable, library versions + per-model timings, a fully executed notebook (outputs embedded, not
  stripped), and McNemar paired significance tests (full k-fold CV remains out of scope at this data
  scale — disclosed, not hidden).